In [23]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [24]:
df = pd.read_csv('train.csv')

In [25]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Dropping Irrelevant Columns

`PassengerId`, `Name`, `Ticket`, and `Cabin` are dropped because:
- `PassengerId` — just a row number, carries no predictive signal
- `Name` / `Ticket` — high-cardinality text, would need complex NLP to be useful
- `Cabin` — ~77% missing values, too sparse to reliably impute

After this drop, the 7 remaining columns are the actual features we'll work with.

In [26]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

## Train-Test Split

Split happens **before any preprocessing** — this is intentional.

Every transformer (imputer, encoder, scaler) must be `fit` only on training data so it never "sees" test set statistics. If we imputed or encoded the full dataset first and then split, the test set's data would influence the learned parameters — that's **data leakage**, and it makes evaluation unrealistically optimistic.

**Correct order always:**
```
Raw data → split → fit all transformers on X_train → transform X_train → transform X_test
```

In [27]:
# train/test/split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

In [28]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [29]:
y_train.head()

331    0
733    0
382    0
704    0
813    0
Name: Survived, dtype: int64

## Identifying Missing Values

Before imputing anything, we check what actually needs imputing.

**Findings:**
- `Age` — **177 missing** (~20% of data). Strategy: mean imputation (`SimpleImputer()` default)
- `Embarked` — **2 missing** (negligible). Strategy: most frequent value imputation

Everything else is complete — no action needed for `Pclass`, `Sex`, `SibSp`, `Parch`, `Fare`.

In [30]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

## Step 1 — Imputation

Two separate `SimpleImputer` objects are created — one per column, because each needs a different strategy.

**Why not one imputer for both?**  
`SimpleImputer` applies the same strategy to every column it's given. We can't pass both `Age` (needs mean) and `Embarked` (needs most_frequent) to a single imputer — the strategies would conflict.

**Why `fit` only on `X_train`?**  
The imputer learns the fill value (mean age, most frequent port) from training data. Both imputers are then used to `transform` the test set using those training-derived values — ensuring no test data bleeds into preprocessing.

In [31]:
# Applying Imputation

si_age = SimpleImputer()
si_embarked = SimpleImputer(strategy='most_frequent')

X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']])
X_test_embarked = si_embarked.transform(X_test[['Embarked']])

## Step 2 — One-Hot Encoding

`Sex` and `Embarked` are nominal categorical columns — no natural order — so they get One-Hot Encoded.

**Why two separate `OneHotEncoder` objects?**  
Notice that `Embarked` is being encoded from `X_train_embarked` (the already-imputed output), **not** directly from `X_train[['Embarked']]`. This is because `Embarked` had 2 missing values — we had to impute it first. Since `ohe_sex` and `ohe_embarked` receive data from different sources (one from the original DataFrame, one from the imputer's output), they must be separate objects.

`sparse_output=False` returns a dense numpy array instead of a sparse matrix.  
`handle_unknown='ignore'` makes the encoder output all-zeros for any unseen category at inference time (rather than crashing).

In [32]:
# one hot encoding Sex and Embarked
# why we didnt used single OneHotEncoder
ohe_sex = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

X_test_sex = ohe_sex.transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.transform(X_test_embarked)

In [33]:
X_train_sex

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]], shape=(712, 2))

## Step 3 — Removing Already-Processed Columns

`Sex`, `Age`, and `Embarked` have been transformed into new columns (`X_train_sex`, `X_train_age`, `X_train_embarked`). The original string/float versions must now be dropped from `X_train_rem` so they aren't double-counted in the final feature matrix.

**This manual bookkeeping is one of the core pain points of working without a Pipeline** — you must mentally track which columns have been processed, which remain, and maintain perfect consistency between training and inference code.

In [34]:
# munually joining
X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'])

X_test_rem = X_test.drop(columns = ['Sex','Age','Embarked'])


## Step 4 — Manual Assembly with `np.concatenate`

All processed pieces are manually stitched together into the final feature matrix.

**Column order of `X_train_transformed`:**
```
[Pclass, SibSp, Parch, Fare]   ← from X_train_rem      (4 cols)
[Age]                           ← from X_train_age       (1 col)
[female, male]                  ← from X_train_sex OHE   (2 cols)
[C, Q, S]                       ← from X_train_embarked OHE (3 cols)
                                                    Total = 10 cols
```

**This is fragile.** The order of columns here must be **exactly replicated** at inference time. Any deviation — wrong index, wrong order, forgetting a column — silently produces wrong predictions with no error raised.

⚠️ Notice `X_test_transformed` is built identically. This exact repetition is another failure point — if you update the training assembly, you must remember to update the test assembly too.

In [35]:
X_train_transformed = np.concatenate((X_train_rem,X_train_age,X_train_sex,X_train_embarked),axis=1)
X_test_transformed = np.concatenate((X_test_rem,X_test_age,X_test_sex,X_test_embarked),axis=1)

## Training and Evaluation

A `DecisionTreeClassifier` is fit on the manually assembled `X_train_transformed`.

In [37]:
clf = DecisionTreeClassifier()
clf.fit(X_train_transformed,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [41]:
y_pred = clf.predict(X_test_transformed)
y_pred

array([0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 1, 1])

**Training accuracy: ~78.2%**

This is the baseline. We'll compare this against the Pipeline approach.

In [42]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7821229050279329

## Saving the Model — The Pickle Problem

To deploy this model (serve predictions on new data), **three separate files** must be saved:
- `ohe_sex.pkl` — the Sex encoder
- `ohe_embarked.pkl` — the Embarked encoder
- `clf.pkl` — the classifier

**Why is this a problem?**

At inference time, whoever loads these files must:
1. Load all 3 pickles
2. Know the exact column order of the input
3. Manually apply `ohe_sex` to the right column, `ohe_embarked` to another
4. Manually concatenate everything in the correct order before calling `clf.predict()`

Miss any of these steps or get the order wrong → wrong predictions, silently.

This is the key motivation for Pipelines — which bundle the entire transformation + prediction chain into **a single serializable object**.

In [43]:
import pickle

In [46]:
# Saves to current directory
pickle.dump(ohe_sex, open('ohe_sex.pkl', 'wb')) 
pickle.dump(ohe_embarked, open('ohe_embarked.pkl', 'wb')) 
pickle.dump(clf, open('clf.pkl', 'wb'))